# Week 5, Day 1 — Agent Foundations
### Reasoning Loops, Tool Calling & Raw Python Agents


## Task 1 — Agent Concepts & Mental Model

### Chatbot vs. Workflow vs. Agent

A **chatbot** mainly answers questions or responds to whatever the user asks. It focuses on generating text but usually doesn't perform actions outside the conversation.

A **workflow** is a predefined sequence of steps. Every step is already decided by the developer, so it always follows the same path regardless of the situation.

An **agent** is more flexible. Instead of following a fixed path, it decides what to do next based on the current situation. It can use tools, observe the results, and continue working until it reaches the goal.

---

### What makes something "agentic"?

An AI system is considered agentic when it can:

- **Autonomy:** Decide its own next action instead of following a fixed script.
- **Tool use:** Use external tools like APIs, calculators, databases, or files instead of relying only on its own knowledge.
- **Multi-step planning:** Break a large task into smaller steps and solve them one by one.
- **Self-correction:** Learn from tool outputs or errors and change its approach if something doesn't work.

---

### ReAct Pattern (Reason → Act → Observe)

The ReAct pattern is a simple loop that many AI agents follow.


         User Goal
             │
             ▼
         ┌────────┐
         │ Reason │
         └────────┘
             │
             ▼
         ┌────────┐
         │  Act   │
         └────────┘
             │
             ▼
         ┌──────────┐
         │ Observe  │
         └──────────┘
             │
             ▼
      Is the task done?
       │            │
     No             Yes
       │             │
       └──────► Final Answer

```
loop:
    Reason:  "Given the goal and what I know so far, what should I do next?"
    Act:     call a tool (or produce a final answer)
    Observe: read the tool's result
    -> feed the observation back into the next Reason step
until: the model produces a final answer (or a safety limit is hit)
```

Pseudocode:
```python
while not done and iterations < max_iterations:
    response = model(messages)              # Reason
    if response.wants_tool_call:
        result = execute_tool(response)      # Act
        messages.append(observation(result))  # Observe
    else:
        done = True                           # final answer
```

**When is an agent overkill?**
Not every problem needs an AI agent. If the task is simple, such as translating text, summarizing a paragraph, or formatting JSON, a single prompt or a normal Python script is usually enough. An agent is more useful when the next step depends on new information, tool outputs, or multiple decisions that cannot be planned beforehand.


## Task 2 — Tool Calling Fundamentals

I have implemented these tools: 
- `calculator` — safe arithmetic evaluator (uses a restricted AST, never raw `eval`)
- `get_weather` — stubbed weather lookup (fakes an external API call)
- `read_text_file` — reads a small local text file (bonus tool)

**Why tool descriptions matter:** the model never sees your source code — the
`description` field *is* the entire interface it reasons over when deciding
whether and how to call a tool. A vague description ("gets weather") invites
the model to call it for the wrong situations or guess at argument formats.
A precise one (scope, expected input format, what it does *not* do) is what
makes tool selection and argument-filling reliable.


In [1]:
"""
Task 2 -- Tool definitions.

Each tool has:
  1. A Python function that actually executes it.
  2. A JSON Schema description (name, description, input_schema/parameters)
     that gets sent to the model so it knows the tool exists and how to call it.
"""

import ast
import operator as op
import os

# --------------------------------------------------------------------------
# 1. Tool implementations
# --------------------------------------------------------------------------

_SAFE_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
    ast.Div: op.truediv, ast.Pow: op.pow, ast.USub: op.neg,
    ast.Mod: op.mod,
}


def _safe_eval(node):
    """Evaluate a restricted arithmetic AST -- never use raw eval() on model input."""
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Unsupported expression")


def calculator(expression: str) -> str:
    """Safely evaluate a basic arithmetic expression."""
    try:
        tree = ast.parse(expression, mode="eval").body
        result = _safe_eval(tree)
        return json.dumps({"expression": expression, "result": result})
    except ZeroDivisionError:
        return json.dumps({"expression": expression, "error": "Error: division by zero"})
    except Exception as e:
        return json.dumps({"expression": expression, "error": f"Error: {e}"})


_FAKE_WEATHER_DB = {
    "lahore": {"tempC": 35, "condition": "Sunny"},
    "islamabad": {"tempC": 29, "condition": "Partly cloudy"},
    "karachi": {"tempC": 32, "condition": "Humid"},
}


def get_weather(city: str) -> str:
    """Stub weather lookup -- fakes a real weather API call for demo purposes."""
    data = _FAKE_WEATHER_DB.get(city.strip().lower())
    if not data:
        return json.dumps({"city": city, "error": f"Error: no weather data for '{city}'"})
    return json.dumps({"city": city, **data})


def read_text_file(path: str) -> str:
    """Read a small local text file and return its contents (truncated)."""
    try:
        if not os.path.exists(path):
            return json.dumps({"path": path, "error": f"Error: file not found: {path}"})
        with open(path, "r") as f:
            content = f.read(2000)
        return json.dumps({"path": path, "content": content})
    except Exception as e:
        return json.dumps({"path": path, "error": f"Error: {e}"})


import json  # noqa: E402  (placed here so calculator()/get_weather() above can use it)


# --------------------------------------------------------------------------
# 2. JSON Schemas (OpenAI tool-calling format: type/function/name/description/parameters)
# --------------------------------------------------------------------------

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": (
                "Evaluate a basic arithmetic expression (+, -, *, /, %, **). "
                "Use this any time the user asks for a numeric calculation. "
                "Input must be a plain math expression string, e.g. '23 * 47'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A valid arithmetic expression, e.g. '(4 + 5) * 2'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": (
                "Look up the current weather for a named city. "
                "Only works for cities in the demo database (Lahore, Islamabad, Karachi). "
                "Returns temperature in Celsius and a short condition string."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. 'Lahore'",
                    }
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_text_file",
            "description": (
                "Read the contents of a small local text file given its path. "
                "Use this only when the user explicitly refers to a local file."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Absolute or relative path to a text file",
                    }
                },
                "required": ["path"],
            },
        },
    },
]

# Registry mapping tool name -> Python callable, used by the agent loop to
# actually execute whatever the model asked for.
TOOL_REGISTRY = {
    "calculator": calculator,
    "get_weather": get_weather,
    "read_text_file": read_text_file,
}


In [7]:
# Setup cell
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("NETIXSOL_API_KEY"),
    base_url="https://llm.netixsol.com/v1"
)

MODEL = "fast"

In [11]:
"""
Task 2 -- Send a single request, let the model choose a tool,
manually execute the tool, then return the result back to the model.
"""

import json

def run_task2_demo():

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant with access to tools."
        },
        {"role": "user", "content": "What is 23 * 47?"},
        # {"role": "user", "content": "What's the weather in Lahore?"},
        # {"role": "user", "content": "Read the file sample.txt"},
    ]

    print("=" * 60)
    print("Task 2 - Tool Calling Demo")
    print("=" * 60)
    print(f"User: {messages[-1]['content']}\n")

    # Step 1: Let the model decide whether to call a tool
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
    )

    msg = response.choices[0].message

    # If the model answers directly instead of calling a tool
    if not msg.tool_calls:
        print("Assistant:")
        print(msg.content)
        return

    # Step 2: Execute the requested tool
    tool_call = msg.tool_calls[0]

    args = json.loads(tool_call.function.arguments)

    print(f"Tool Selected : {tool_call.function.name}")
    print(f"Arguments     : {args}")

    tool_fn = TOOL_REGISTRY[tool_call.function.name]
    result = tool_fn(**args)

    print("\nTool Result:")
    print(result)

    # Step 3: Send the tool result back to the model
    messages.append(
        {
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments,
                    },
                }
            ],
        }
   )

    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result,
    })

    try:
        final_response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
        )

        print("\nFinal Assistant Response:")
        print(final_response.choices[0].message.content)

    except Exception as e:
        print("\nTool execution completed successfully.")
        print("Unable to generate the final response due to API limitations.")
        print(e)


run_task2_demo()

Task 2 - Tool Calling Demo
User: What is 23 * 47?

Tool Selected : calculator
Arguments     : {'expression': '23 * 47'}

Tool Result:
{"expression": "23 * 47", "result": 1081}

Final Assistant Response:
23 × 47 = 1081.


During testing, I also verified that the get_weather and read_text_file tools were correctly selected by the model for relevant prompts. For the final demonstration, I kept the calculator example because it provides the simplest illustration of the complete tool-calling workflow.

## Task 3 — Build a Minimal Agent Loop

`send message → check for tool_calls → execute → append tool_result → repeat`,
bounded by `max_iterations` so a model that keeps calling tools forever can't
hang the process.


In [12]:
"""
Task 3 -- Minimal ReAct-style agent loop.
Task 4 -- Memory & state handling (conversation memory vs. working memory) + logging.
"""

import json

class Agent:
    """
    A minimal while-loop agent:
        1. Send the user's message to the model.
        2. If the model requests a tool, execute it.
        3. Append the tool result.
        4. Repeat until the model returns a final answer.
        5. Stop after max_iterations to avoid infinite loops.
    """

    def __init__(self, system_prompt: str, model: str = MODEL, max_iterations: int = 6):
        self.client = client
        self.model = model
        self.max_iterations = max_iterations

        # Conversation memory:
        # Stores the full message history sent back to the model each turn.
        self.messages = [
            {
                "role": "system",
                "content": system_prompt
            }
        ]

        # Working memory:
        # Stores the agent's internal state for debugging/control.
        self.scratchpad = {
            "tool_calls_made": [],
            "iterations": 0
        }

    def _log_step(self, label, content):
        print(f"[{label.upper()}] {content}")

    def run(self, user_message: str):

        self.messages.append(
            {
                "role": "user",
                "content": user_message
            }
        )

        self._log_step("user", user_message)

        for i in range(self.max_iterations):

            print("\n" + "=" * 50)
            print(f"Iteration {i+1}")
            print("=" * 50)

            self.scratchpad["iterations"] = i + 1

            response = self.client.chat.completions.create(
                model=self.model,
                messages=self.messages,
                tools=TOOLS,
                tool_choice="auto",
            )

            msg = response.choices[0].message

            # --------------------------------------------------------
            # Model wants to use one or more tools
            # --------------------------------------------------------
            if msg.tool_calls:

                self.messages.append(
                    {
                        "role": "assistant",
                        "content": msg.content,
                        "tool_calls": [
                            {
                                "id": tc.id,
                                "type": "function",
                                "function": {
                                    "name": tc.function.name,
                                    "arguments": tc.function.arguments,
                                },
                            }
                            for tc in msg.tool_calls
                        ],
                    }
                )

                for tool_call in msg.tool_calls:

                    name = tool_call.function.name

                    try:
                        args = json.loads(tool_call.function.arguments)
                    except json.JSONDecodeError:
                        args = {}

                    self._log_step(
                        "reasoning",
                        f"Iteration {i+1}: model wants to call '{name}'"
                    )

                    self._log_step(
                        "tool_call",
                        f"{name}({args})"
                    )

                    if name not in TOOL_REGISTRY:

                        result = json.dumps(
                            {
                                "error": f"Tool '{name}' is not registered."
                            }
                        )

                    else:

                        result = TOOL_REGISTRY[name](**args)

                    self._log_step("observation", result)

                    self.scratchpad["tool_calls_made"].append(
                        {
                            "name": name,
                            "args": args,
                        }
                    )

                    self.messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": result,
                        }
                    )

                continue

            # --------------------------------------------------------
            # Final answer
            # --------------------------------------------------------
            self.messages.append(
                {
                    "role": "assistant",
                    "content": msg.content,
                }
            )

            self._log_step("final_answer", msg.content)

            return msg.content

        # --------------------------------------------------------
        # Guardrail
        # --------------------------------------------------------
        warning = (
            f"Stopped after {self.max_iterations} iterations "
            f"without reaching a final answer."
        )

        self._log_step("guardrail", warning)

        return warning

**Test on a multi-step task requiring 2+ tool calls:**

In [13]:
agent = Agent(system_prompt="You are a helpful assistant with access to tools. "
                             "Use them whenever they'd help answer the question.")
answer = agent.run("Compare the weather in Lahore and Karachi and tell me which city is warmer.")
print("\n=== FINAL ANSWER ===")
print(answer)

[USER] Compare the weather in Lahore and Karachi and tell me which city is warmer.

Iteration 1
[REASONING] Iteration 1: model wants to call 'get_weather'
[TOOL_CALL] get_weather({'city': 'Lahore'})
[OBSERVATION] {"city": "Lahore", "tempC": 35, "condition": "Sunny"}

Iteration 2
[REASONING] Iteration 2: model wants to call 'get_weather'
[TOOL_CALL] get_weather({'city': 'Karachi'})
[OBSERVATION] {"city": "Karachi", "tempC": 32, "condition": "Humid"}

Iteration 3
[FINAL_ANSWER] Based on the current data:

- **Lahore:** 35 °C, sunny  
- **Karachi:** 32 °C, humid  

**Lahore is warmer than Karachi** by about 3 °C.

=== FINAL ANSWER ===
Based on the current data:

- **Lahore:** 35 °C, sunny  
- **Karachi:** 32 °C, humid  

**Lahore is warmer than Karachi** by about 3 °C.


## Task 4 — Memory & State Handling

An agent keeps track of two different types of memory while solving a task.

**Conversation memory** is simply the message history that gets sent back to the model every time we make an API call. Since LLMs are stateless, they only remember what we include in this list. If we don't append previous messages, tool calls, and tool results, the model forgets what happened before.

**Working memory** is the state that our Python program keeps for itself while the task is running. It is not sent to the model. I used it to store the current iteration number and a list of tool calls made during execution. This helps with debugging and controlling the agent without increasing token usage.

I also added simple logging that prints every reasoning step, tool call, observation, and the final answer. This makes it much easier to understand what the agent is doing internally and is a useful debugging habit before using higher-level frameworks.

In [14]:
print("=" * 60)
print("Conversation Memory")
print("=" * 60)

for i, message in enumerate(agent.messages, start=1):
    print(f"\nMessage {i}")
    print(message)

print("\n")

print("=" * 60)
print("Working Memory")
print("=" * 60)

print(f"Iterations completed : {agent.scratchpad['iterations']}")
print("Tool calls made:")

for tool in agent.scratchpad["tool_calls_made"]:
    print(f" - {tool['name']} {tool['args']}")

Conversation Memory

Message 1
{'role': 'system', 'content': "You are a helpful assistant with access to tools. Use them whenever they'd help answer the question."}

Message 2
{'role': 'user', 'content': 'Compare the weather in Lahore and Karachi and tell me which city is warmer.'}

Message 3
{'role': 'assistant', 'content': None, 'tool_calls': [{'id': 'af61b3557', 'type': 'function', 'function': {'name': 'get_weather', 'arguments': '{"city":"Lahore"}'}}]}

Message 4
{'role': 'tool', 'tool_call_id': 'af61b3557', 'content': '{"city": "Lahore", "tempC": 35, "condition": "Sunny"}'}

Message 5
{'role': 'assistant', 'content': None, 'tool_calls': [{'id': '411607750', 'type': 'function', 'function': {'name': 'get_weather', 'arguments': '{"city":"Karachi"}'}}]}

Message 6
{'role': 'tool', 'tool_call_id': '411607750', 'content': '{"city": "Karachi", "tempC": 32, "condition": "Humid"}'}

Message 7
{'role': 'assistant', 'content': 'Based on the current data:\n\n- **Lahore:** 35\u202f°C, sunny  \

## Task 5 — Failure Modes & Guardrails

To better understand how agents behave, I deliberately tested my agent with inputs that could potentially cause failures.

The following three scenarios were tested:

- **Scenario A:** An ambiguous request ("Can you book it for me?")
- **Scenario B:** A tool that returns an error ("What is 10 / 0?")
- **Scenario C:** A request that would normally require a tool that doesn't exist ("Please send an email to my team.")

The goal was to observe how the agent behaves in situations where it cannot confidently complete the task and to identify possible guardrails that make the agent more reliable.

During testing, the agent handled all three situations safely. Instead of blindly calling tools, it either asked for clarification or responded directly when tool usage was unnecessary. This demonstrates that modern LLMs can sometimes avoid inappropriate tool calls on their own.

In [15]:
def scenario_a():
    print("=" * 60)
    print("Scenario A - Ambiguous Request")
    print("=" * 60)

    agent = Agent(
        system_prompt="You are a helpful assistant with access to tools."
    )

    result = agent.run("Can you book it for me?")

    print("\nResult:")
    print(result)


def scenario_b():
    print("=" * 60)
    print("Scenario B - Tool Error")
    print("=" * 60)

    agent = Agent(
        system_prompt="You are a helpful assistant with access to tools."
    )

    result = agent.run("What is 10 / 0?")

    print("\nResult:")
    print(result)


def scenario_c():
    print("=" * 60)
    print("Scenario C - Undefined Tool")
    print("=" * 60)

    agent = Agent(
        system_prompt="You are a helpful assistant with access to tools."
    )

    result = agent.run(
        "Please send an email to my team about the project delay."
    )

    print("\nResult:")
    print(result)


scenario_a()
scenario_b()
scenario_c()

Scenario A - Ambiguous Request
[USER] Can you book it for me?

Iteration 1
[FINAL_ANSWER] I’m not able to make reservations or purchases directly, but I can certainly help you with the process! Could you let me know what you’d like to book (e.g., a flight, a hotel, a restaurant reservation, etc.) and any details such as dates, location, or preferences? With that information I can guide you through the steps, find options, and give you the links or phone numbers you’ll need to complete the booking.

Result:
I’m not able to make reservations or purchases directly, but I can certainly help you with the process! Could you let me know what you’d like to book (e.g., a flight, a hotel, a restaurant reservation, etc.) and any details such as dates, location, or preferences? With that information I can guide you through the steps, find options, and give you the links or phone numbers you’ll need to complete the booking.
Scenario B - Tool Error
[USER] What is 10 / 0?

Iteration 1
[FINAL_ANSWER] 

## Interpretation:
### Scenario A
Observation: The request was ambiguous, so the agent did not call any tool. Instead, it asked the user for more information before proceeding. This is desirable behavior because it avoids making incorrect assumptions.

### Scenario B
Observation: Instead of calling the calculator tool, the model answered directly using its own reasoning. This shows that LLMs may sometimes choose not to use available tools if they believe they already know the answer.

### Scenario C
Observation: The model did not hallucinate a tool call. Instead, it recognized that sending emails was outside its capabilities and offered to help draft the email instead.

| Failure Mode | What happens | Mitigation |
|--------------|-------------|------------|
| Infinite loop | The agent keeps calling tools forever | Set a `max_iterations` limit |
| Undefined tool | During testing, the model recognized it could not send emails and instead offered to draft one instead of hallucinating a tool call. | Clearly describe available tools and restrict execution to registered tools only. |
| Invalid tool arguments | The model sends incorrect or incomplete arguments | Validate JSON and required fields |
| Tool execution error | A tool fails (e.g. divide by zero) | Return a structured error instead of crashing |
| Ambiguous request | The user doesn't provide enough information | Ask a clarifying question instead of guessing |
| High API cost | Many iterations increase latency and cost | Keep iteration limits and choose smaller models when appropriate |

### Why do frameworks like LangChain, LangGraph, and CrewAI exist?

Building a simple agent by hand helped me understand what actually happens behind the scenes. Even though the core idea is straightforward, there is a lot of repetitive code for managing tool calls, conversation history, memory, logging, error handling, and guardrails.

Frameworks such as LangChain, LangGraph, and CrewAI provide these features out of the box, making it much easier to build larger and more complex AI applications. Since I first implemented the agent manually, I now have a much better understanding of what these frameworks are doing internally instead of treating them as "magic."